In [7]:
#| default_exp multihop

In [8]:
#| export
import torch, numpy as np
from typing import Dict, List, Optional
from fastcore.utils import patch

from xcai.learner import XCLearner
from transformers.trainer_pt_utils import nested_concat

In [9]:
#| export
class MultihopBeamHypotheses:
    """
    Tracks and manages hypotheses (useful as an alternative API or for custom extensions).
    """
    def __init__(self, num_beams: int):
        self.num_beams = num_beams
        self.beams = []
        self.worst_score = -1e9
        
    def add(self, path: torch.Tensor, score: float):
        if len(self.beams) < self.num_beams or score > self.worst_score:
            self.beams.append((score, path))
            if len(self.beams) > self.num_beams:
                sorted_beams = sorted(enumerate(self.beams), key=lambda x: x[1][0])
                del self.beams[sorted_beams[0][0]]
            self.worst_score = min(b[0] for b in self.beams)
            
    def is_done(self):
        return len(self.beams) >= self.num_beams


class MultihopBeamSearchScorer:
    """
    Vectorized beam search scorer specifically optimized for dense multihop document retrieval.
    """
    def __init__(self, batch_size: int, num_beams: int, device: torch.device):
        self.batch_size = batch_size
        self.num_beams = num_beams
        self.device = device
        self.beam_scores = torch.zeros((batch_size, num_beams), dtype=torch.float, device=device)
        self.beam_scores[:, 1:] = -1e9
        
    def process(self, next_scores: torch.Tensor, next_docs: torch.Tensor, beam_paths: torch.Tensor, current_topk: int):
        batch_size = next_scores.shape[0]
        current_beam_size = next_scores.shape[1]
        active_beam_scores = self.beam_scores[:, :current_beam_size]
        next_scores = next_scores + active_beam_scores.unsqueeze(-1)
        
        next_scores = next_scores.view(batch_size, -1)
        k = min(self.num_beams, next_scores.shape[1])
        topk_scores, topk_indices = torch.topk(next_scores, k, dim=-1)
        
        self.beam_scores = torch.zeros((batch_size, self.num_beams), dtype=torch.float, device=self.device)
        self.beam_scores[:, :k] = topk_scores
        self.beam_scores[:, k:] = -1e9
        
        beam_indices = torch.div(topk_indices, current_topk, rounding_mode="floor")
        
        next_docs_flat = next_docs.contiguous().view(batch_size, -1)
        selected_docs = torch.gather(next_docs_flat, 1, topk_indices)
        
        if beam_paths.shape[-1] == 0:
            new_paths = selected_docs.unsqueeze(-1)
        else:
            gathered_paths = torch.gather(
                beam_paths, 1, beam_indices.unsqueeze(-1).expand(-1, -1, beam_paths.shape[-1])
            )
            new_paths = torch.cat([gathered_paths, selected_docs.unsqueeze(-1)], dim=-1)
            
        return topk_scores, selected_docs, new_paths, beam_indices
        
    def finalize(self, beam_paths, topk_scores):
        return beam_paths.cpu(), topk_scores.cpu()
        

In [10]:
#| export
class MultihopLearner(XCLearner):
    
    def evaluation_loop(
        self,
        dataloader,
        beam_size: int = 5,
        num_hops: int = 3,
        topk_per_hop: int = 10,
        predict_with_generation: bool = False,
        predict_with_representation: bool = True,
    ):
        """
        Efficient vectorized multi-hop beam search retrieval.
        """
        model = self._wrap_model(self.model, training=False, dataloader=dataloader)
        
        if len(self.accelerator._models) == 0 and model is self.model:
            model = self.accelerator.prepare_model(model, evaluation_mode=True)
            
        model.eval()
        
        all_final_paths = []
        all_final_scores = []
        all_label_ids = []
        
        for step, inputs in enumerate(dataloader):
            base_queries = inputs.get('data_input_text', inputs.get('data_identifier', []))
            batch_size = len(base_queries)
            
            labels = inputs.get("labels", inputs.get("data_lbl2data_idx", None))
            if labels is not None:
                all_label_ids.append(labels.detach().cpu())
            
            scorer = MultihopBeamSearchScorer(batch_size, beam_size, model.device)
            beam_paths = torch.zeros((batch_size, beam_size, 0), dtype=torch.long, device=model.device)
            current_queries = [[q] for q in base_queries]
            current_inputs = inputs
            
            for hop in range(num_hops):
                loss, output = self.prediction_step(
                    model, 
                    current_inputs, 
                    prediction_loss_only=False,
                    predict_with_generation=predict_with_generation,
                    predict_with_representation=predict_with_representation
                )
                
                pred_idx = output["pred_idx"]
                pred_score = output["pred_score"]
                
                N = len(current_queries) * len(current_queries[0])
                
                if pred_idx.dim() == 1:
                    topk = len(pred_idx) // N
                    pred_idx = pred_idx.view(batch_size, -1, topk)
                    pred_score = pred_score.view(batch_size, -1, topk)
                else:
                    topk = pred_idx.shape[-1]
                    pred_idx = pred_idx.view(batch_size, -1, topk)
                    pred_score = pred_score.view(batch_size, -1, topk)
                    
                current_topk = min(topk, topk_per_hop)
                pred_idx = pred_idx[:, :, :current_topk]
                pred_score = pred_score[:, :, :current_topk]
                
                topk_scores, selected_docs, beam_paths, beam_indices = scorer.process(
                    pred_score, pred_idx, beam_paths, current_topk
                )
                
                k = topk_scores.shape[1]
                
                if hop < num_hops - 1:
                    new_queries = []
                    for b in range(batch_size):
                        batch_new_queries = []
                        for h_idx in range(k):
                            prev_beam = beam_indices[b, h_idx].item()
                            doc_id = selected_docs[b, h_idx].item()
                            prev_query = current_queries[b][prev_beam]
                            # Assume expand_query is implemented on the learner
                            expanded = self.expand_query(prev_query, doc_id)
                            batch_new_queries.append(expanded)
                        new_queries.append(batch_new_queries)
                    
                    current_queries = new_queries
                    flat_queries = [q for batch_q in current_queries for q in batch_q]
                    
                    tokenizer = getattr(self, "tokenizer", None)
                    if tokenizer is None and hasattr(dataloader.dataset, "tokenizer"):
                        tokenizer = dataloader.dataset.tokenizer
                    elif tokenizer is None:
                        raise RuntimeError("Tokenizer not found on XCLearner or Dataset.")
                        
                    tokenized = tokenizer(
                        flat_queries, 
                        padding=True, 
                        truncation=True, 
                        max_length=self.args.max_length if hasattr(self.args, "max_length") else 512,
                        return_tensors="pt"
                    )
                    tokenized = {k_tok: v_tok.to(model.device) for k_tok, v_tok in tokenized.items()}
                    
                    current_inputs = {
                        "data_input_ids": tokenized["input_ids"],
                        "data_attention_mask": tokenized["attention_mask"],
                    }
                    if "token_type_ids" in tokenized:
                        current_inputs["data_token_type_ids"] = tokenized["token_type_ids"]

            final_paths, final_scores = scorer.finalize(beam_paths, topk_scores)
            all_final_paths.append(final_paths)
            all_final_scores.append(final_scores)
            
        return {
            "paths": torch.cat(all_final_paths, dim=0) if all_final_paths else None,
            "scores": torch.cat(all_final_scores, dim=0) if all_final_scores else None,
            "label_ids": torch.cat(all_label_ids, dim=0) if all_label_ids else None
        }

    def predict(
        self,
        test_dataset,
        ignore_keys: Optional[List[str]] = None,
        metric_key_prefix: str = "test",
        **gen_kwargs
    ):
        dataloader = self.get_test_dataloader(test_dataset)
        output = self.evaluation_loop(dataloader, **gen_kwargs)
        
        paths = output.get("paths")
        label_ids = output.get("label_ids")
        
        metrics = {}
        if paths is not None and label_ids is not None and self.compute_metrics is not None:
            num_hops = paths.shape[-1]
            for hop in range(num_hops):
                hop_preds = paths[:, :, hop]
                
                hop_eval = EvalPrediction(
                    predictions=hop_preds.numpy() if isinstance(hop_preds, torch.Tensor) else hop_preds, 
                    label_ids=label_ids.numpy() if isinstance(label_ids, torch.Tensor) else label_ids
                )
                
                hop_metrics = self.compute_metrics(hop_eval)
                
                for k, v in hop_metrics.items():
                    metrics[f"{metric_key_prefix}_hop_{hop}_{k}"] = v
                    
        return PredictionOutput(predictions=paths, label_ids=label_ids, metrics=metrics)

    def evaluate(
        self,
        eval_dataset = None,
        ignore_keys: Optional[List[str]] = None,
        metric_key_prefix: str = "eval",
        **gen_kwargs
    ):
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        output = self.predict(
            test_dataset=eval_dataset,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
            **gen_kwargs
        )
        
        self.log(output.metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, output.metrics)
        return output.metrics
